In [17]:
# imporitng libraries
import os
import joblib
import torch
import pyiqa
import numpy as np
import pandas as pd

from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [18]:
# configuring paths
ROOT_IMAGES = "dataset_split"
ROOT_EMBEDDINGS = "embeddings"

In [19]:
# loading our traineed svm model
svm = joblib.load("svm_rejection_model.pkl")
scaler = joblib.load("svm_scaler.pkl")
print("SVM Loaded")

SVM Loaded


In [20]:
# loading MUSIQ for quality check of test images
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
musiq_metric = pyiqa.create_metric("musiq", device=device)
print("MUSIQ Loaded")

cpu
Loading pretrained model MUSIQ from /Users/admin/.cache/torch/hub/pyiqa/musiq_koniq_ckpt-e95806b9.pth
MUSIQ Loaded


In [21]:
# quality score function
def get_quality_score(image_path):
    score = musiq_metric(image_path)
    return float(score.cpu().item())

In [22]:
# loading gallery database
def load_gallery_db(gallery_root):
    gallery_db = {}
    total = 0

    for identity in os.listdir(gallery_root):
        identity_dir = os.path.join(gallery_root, identity)
        gallery_db[identity] = []

        for file in os.listdir(identity_dir):
            emb = np.load(os.path.join(identity_dir, file))
            gallery_db[identity].append(emb)
            total += 1

    print("Gallery Embeddings Loaded:", total)
    return gallery_db

In [23]:
# gallery search
def search_gallery(probe_embedding, gallery_db):
    identity_scores = {}

    for identity in gallery_db:
        sims = []

        for gallery_emb in gallery_db[identity]:
            sim = cosine_similarity(
                probe_embedding.reshape(1, -1), gallery_emb.reshape(1, -1)
            )[0][0]
            sims.append(sim)
        identity_scores[identity] = max(sims)

    sorted_scores = sorted(identity_scores.items(), key=lambda x: x[1], reverse=True)
    predicted_identity = sorted_scores[0][0]
    best_similarity = sorted_scores[0][1]
    second_similarity = sorted_scores[1][1]
    margin = best_similarity - second_similarity

    return (predicted_identity, best_similarity, second_similarity, margin)

In [24]:
# loading test gallery
gallery_db = load_gallery_db(os.path.join(ROOT_EMBEDDINGS, "test", "gallery"))

Gallery Embeddings Loaded: 482


In [25]:
# testing
probe_emb_root = os.path.join(ROOT_EMBEDDINGS, "test", "degraded_probes")
probe_img_root = os.path.join(ROOT_IMAGES, "test", "degraded_probes")
results = []
counter = 0

for identity in tqdm(os.listdir(probe_emb_root)):
    emb_dir = os.path.join(probe_emb_root, identity)
    img_dir = os.path.join(probe_img_root, identity)

    for emb_file in os.listdir(emb_dir):
        emb_path = os.path.join(emb_dir, emb_file)
        probe_embedding = np.load(emb_path)
        base_name = os.path.splitext(emb_file)[0]
        image_path = None

        for ext in [".jpg", ".jpeg", ".png"]:
            candidate = os.path.join(img_dir, base_name + ext)

            if os.path.exists(candidate):
                image_path = candidate
                break

        if image_path is None:
            continue

        quality_score = get_quality_score(image_path)
        predicted_identity, best_similarity, second_similarity, margin = search_gallery(
            probe_embedding, gallery_db
        )

        import pandas as pd
        features=pd.DataFrame([[quality_score,best_similarity,margin]],
                              columns=["quality_score", "best_similarity", "margin" ])
        features_scaled = scaler.transform(features)
        svm_decision = svm.predict(features_scaled)[0]
        label = int(predicted_identity == identity)

        results.append(
            {
                "probe_image": base_name,
                "true_identity": identity,
                "predicted_identity": predicted_identity,
                "quality_score": quality_score,
                "best_similarity": best_similarity,
                "second_best_similarity": second_similarity,
                "margin": margin,
                "svm_decision": svm_decision,
                "label": label,
            }
        )

        counter += 1

        if counter % 50 == 0:
            print(f"\nProcessed {counter}")
            print("Truth:", identity)
            print("Prediction:", predicted_identity)
            print("Quality:", round(quality_score, 4))
            print("Similarity:", round(best_similarity, 4))
            print("Margin:", round(margin, 4))
            print("Accept?" if svm_decision == 1 else "Reject?")

 10%|█         | 3/30 [00:09<01:19,  2.96s/it]


Processed 50
Truth: Angelina_Jolie
Prediction: Angelina_Jolie
Quality: 30.7015
Similarity: 0.6188
Margin: 0.4595
Accept?


 23%|██▎       | 7/30 [00:29<01:37,  4.24s/it]


Processed 100
Truth: Carlos_Moya
Prediction: Carlos_Moya
Quality: 27.5332
Similarity: 0.5668
Margin: 0.3876
Accept?


 33%|███▎      | 10/30 [00:39<01:11,  3.56s/it]

...

 80%|████████  | 24/30 [01:34<00:29,  5.00s/it]


Processed 450
Truth: George_Robertson
Prediction: George_Robertson
Quality: 16.372
Similarity: 0.7661
Margin: 0.5958
Accept?


 90%|█████████ | 27/30 [01:43<00:10,  3.43s/it]


Processed 500
Truth: Megawati_Sukarnoputri
Prediction: Megawati_Sukarnoputri
Quality: 20.0234
Similarity: 0.7018
Margin: 0.4913
Accept?


100%|██████████| 30/30 [01:53<00:00,  3.79s/it]


In [26]:
# saving predictions
test_df = pd.DataFrame(results)
test_df.to_csv("test_predictions.csv", index=False)
print(test_df.shape)
print("Saved test_predictions.csv")

(540, 9)
Saved test_predictions.csv


In [27]:
# overall accuracy
identity_accuracy = test_df["label"].mean()
print("Identity Accuracy:", identity_accuracy)

Identity Accuracy: 0.9777777777777777


In [28]:
# svm performance
y_true = test_df["label"]
y_pred = test_df["svm_decision"]

print("Accept/Reject Accuracy:", accuracy_score(y_true, y_pred))
print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred))

Accept/Reject Accuracy: 0.9740740740740741
[[ 11   1]
 [ 13 515]]
              precision    recall  f1-score   support

           0       0.46      0.92      0.61        12
           1       1.00      0.98      0.99       528

    accuracy                           0.97       540
   macro avg       0.73      0.95      0.80       540
weighted avg       0.99      0.97      0.98       540

